# Ring Minimization — L-BFGS Phase Diagrams

For each n ∈ {3, 4, 5, 6, 7, 8} and each point in the parameter grid
(opening_angle, E_AB), minimise the ring configuration energy using L-BFGS
and visualise stability / structural validity as heatmaps.

Patch convention: patch A at body-frame 0 → lab angle θ; patch B at body-frame α → lab angle θ + α.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import jax
import jax.numpy as jnp
import scipy.optimize
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from tqdm.auto import tqdm

# Enable 64-bit precision (required for accurate Hessians)
jax.config.update('jax_enable_x64', True)

from config_patchy_particle import (
    YIELD_ALPHAS_DEG, YIELD_ZETAS,
    ALPHA, PATCH_SIZE, CENTER_RADIUS,
)
from energy_functions import total_energy

## Parameter grid — aligned with cluster simulation config

In [ ]:
# Parameter grid matches run_yield_simulation_cluster.py
opening_angles_deg = YIELD_ALPHAS_DEG[::2]                  # np.linspace(70, 115, 10)
E_AB_values        = 2.0 * YIELD_ZETAS[4:7]                 # logspace(0.1..50)*2

N_ANGLE = len(opening_angles_deg)
N_EAB   = len(E_AB_values)

# Fixed Morse parameters
alpha_morse = float(ALPHA)   # 7
E_AA = 1.0
E_BB = 1.0
r_patch = float(CENTER_RADIUS)  # 1.0
rep_A = 10.0
patch_sigma = float(PATCH_SIZE)

N_RINGS = [3, 4, 5, 6, 7, 8]

print(f"Opening angles (deg): {np.round(opening_angles_deg, 1)}")
print(f"E_AB values:          {np.round(E_AB_values, 2)}")
print(f"Grid size: {N_ANGLE} x {N_EAB}")

## Helper functions

In [ ]:
from config_patchy_particle import regular_ngon_reference


def build_q0(n, opening_angle_rad, r_patch=1.0):
    """
    Build the flat initial state vector q0 = [x1,y1,theta1, ...] for an n-ring.

    Circumradius R chosen so the nearest-neighbour center-to-center distance
    equals 2*r_patch (hard-sphere contact), giving patches near equilibrium.
    """
    R = r_patch / np.sin(np.pi / n)
    positions, orientations, _ = regular_ngon_reference(n, R=R,
                                                         opening_angle_rad=opening_angle_rad)
    q0 = np.zeros(3 * n)
    q0[0::3] = positions[:, 0]
    q0[1::3] = positions[:, 1]
    q0[2::3] = orientations
    return q0


def patch_positions_from_q(q, opening_angle, r_patch=1.0):
    """Return pA (n,2) and pB (n,2) for a given state vector."""
    n = len(q) // 3
    q_r = q.reshape(n, 3)
    centers = q_r[:, :2]
    thetas  = q_r[:, 2]
    pA = centers + r_patch * np.column_stack([np.cos(thetas), np.sin(thetas)])
    pB = centers + r_patch * np.column_stack([np.cos(thetas + opening_angle),
                                               np.sin(thetas + opening_angle)])
    return pA, pB


def check_validity(q_min, opening_angle, n, r_patch=1.0):
    """
    Validity check: for each consecutive pair (i, i+1 mod n),
    verify ||pB_i - pA_{i+1}|| < 2*r_patch  (B->A directed ring topology).
    Returns 1 if all n pairs pass, 0 otherwise.
    """
    pA, pB = patch_positions_from_q(q_min, opening_angle, r_patch)
    for i in range(n):
        j = (i + 1) % n
        dist = np.linalg.norm(pB[i] - pA[j])
        if dist >= 2 * r_patch:
            return 0
    return 1

## L-BFGS minimization over the parameter grid

In [ ]:
def minimize_ring(n, opening_angle_rad, E_AB):
    """
    Minimise the n-ring energy for given opening_angle and E_AB.
    Returns (q_min, n_negative_modes, grad_magnitude, validity, success).
    """
    q0 = build_q0(n, opening_angle_rad, r_patch)

    def f(q):
        return float(total_energy(
            jnp.array(q), opening_angle_rad, E_AB, alpha_morse,
            E_AA=E_AA, E_BB=E_BB, r_patch=r_patch,
            rep_A=rep_A, patch_sigma=patch_sigma,
        ))

    grad_f = jax.grad(lambda q: total_energy(
        q, opening_angle_rad, E_AB, alpha_morse,
        E_AA=E_AA, E_BB=E_BB, r_patch=r_patch,
        rep_A=rep_A, patch_sigma=patch_sigma,
    ))

    def jac(q):
        return np.array(grad_f(jnp.array(q)))

    result = scipy.optimize.minimize(
        f, q0, jac=jac, method='L-BFGS-B',
        options={'ftol': 1e-12, 'gtol': 1e-8, 'maxiter': 10000},
    )
    q_min = result.x

    # Gradient magnitude at minimum
    g = jac(q_min)
    grad_mag = float(np.linalg.norm(g))

    # Hessian eigenvalues
    hess_fn = jax.hessian(lambda q: total_energy(
        q, opening_angle_rad, E_AB, alpha_morse,
        E_AA=E_AA, E_BB=E_BB, r_patch=r_patch,
        rep_A=rep_A, patch_sigma=patch_sigma,
    ))
    H = np.array(hess_fn(jnp.array(q_min)))
    eigenvalues = np.linalg.eigvalsh(H)
    n_negative = int(np.sum(eigenvalues < -1e-6))

    # Structural validity
    valid = check_validity(q_min, opening_angle_rad, n, r_patch)

    return q_min, n_negative, grad_mag, valid, bool(result.success)


# Quick sanity check on n=3
_q, _neg, _gm, _v, _ok = minimize_ring(3, np.deg2rad(opening_angles_deg[0]), E_AB_values[0])
print(f"Sanity check n=3: n_negative={_neg}, grad_mag={_gm:.2e}, valid={_v}, success={_ok}")

## Run full sweep and save results

In [ ]:
for n in N_RINGS:
    print(f"\n=== n = {n} ring ===")
    q_min_grid         = np.zeros((N_ANGLE, N_EAB, 3 * n))
    n_negative_modes   = np.zeros((N_ANGLE, N_EAB), dtype=int)
    grad_magnitude     = np.zeros((N_ANGLE, N_EAB))
    validity           = np.zeros((N_ANGLE, N_EAB), dtype=int)
    minimization_success = np.zeros((N_ANGLE, N_EAB), dtype=bool)

    for ai, alpha_deg in enumerate(tqdm(opening_angles_deg, desc=f"n={n} angles")):
        opening_angle_rad = np.deg2rad(alpha_deg)
        for ei, E_AB in enumerate(E_AB_values):
            q_min, neg, gm, v, ok = minimize_ring(n, opening_angle_rad, E_AB)
            q_min_grid[ai, ei]           = q_min
            n_negative_modes[ai, ei]     = neg
            grad_magnitude[ai, ei]       = gm
            validity[ai, ei]             = v
            minimization_success[ai, ei] = ok

    np.savez(
        f"minimized_n{n}_ring.npz",
        q_min_grid=q_min_grid,
        n_negative_modes=n_negative_modes,
        grad_magnitude=grad_magnitude,
        validity=validity,
        opening_angles_deg=opening_angles_deg,
        E_AB_values=E_AB_values,
        minimization_success=minimization_success,
    )
    print(f"  Saved minimized_n{n}_ring.npz")

## Phase-diagram heatmaps

In [ ]:
def plot_ring_heatmaps(n):
    """Plot three diagnostic heatmaps for an n-ring, loading from the saved .npz file."""
    data = np.load(f"minimized_n{n}_ring.npz")
    angles_deg = data['opening_angles_deg']   # (N_ANGLE,)  — x axis
    EAB_vals   = data['E_AB_values']           # (N_EAB,)    — y axis
    neg_modes  = data['n_negative_modes']      # (N_ANGLE, N_EAB)
    grad_mag   = data['grad_magnitude']
    valid      = data['validity']

    # imshow expects [row, col] = [y, x] → transpose so angle is x-axis
    neg_T   = neg_modes.T     # (N_EAB, N_ANGLE)
    grad_T  = np.log10(grad_mag.T + 1e-14)
    valid_T = valid.T

    extent = [angles_deg[0], angles_deg[-1], EAB_vals[0], EAB_vals[-1]]

    fig, axes = plt.subplots(3, 1, figsize=(7, 11))
    fig.suptitle(f"n = {n} ring", fontsize=14, y=0.98)

    # 1. Negative Hessian modes
    ax = axes[0]
    im = ax.imshow(neg_T, origin='lower', aspect='auto', extent=extent,
                   cmap='Reds', interpolation='nearest')
    plt.colorbar(im, ax=ax, label='# negative modes')
    for ai in range(len(angles_deg)):
        for ei in range(len(EAB_vals)):
            ax.text(angles_deg[ai], EAB_vals[ei], str(neg_modes[ai, ei]),
                    ha='center', va='center', fontsize=7, color='black')
    ax.set_xlabel('Opening angle (deg)')
    ax.set_ylabel('E_AB')
    ax.set_title(f'n={n} ring — Negative Hessian modes')

    # 2. Gradient magnitude (log scale)
    ax = axes[1]
    im = ax.imshow(grad_T, origin='lower', aspect='auto', extent=extent,
                   cmap='viridis', interpolation='nearest')
    plt.colorbar(im, ax=ax, label='log₁₀(‖∇E‖)')
    ax.set_xlabel('Opening angle (deg)')
    ax.set_ylabel('E_AB')
    ax.set_title(f'n={n} ring — ‖∇E‖ at minimum')

    # 3. Structural validity
    ax = axes[2]
    cmap_valid = mcolors.ListedColormap(['red', 'green'])
    im = ax.imshow(valid_T, origin='lower', aspect='auto', extent=extent,
                   cmap=cmap_valid, vmin=0, vmax=1, interpolation='nearest')
    plt.colorbar(im, ax=ax, label='valid (1=yes, 0=no)', ticks=[0, 1])
    ax.set_xlabel('Opening angle (deg)')
    ax.set_ylabel('E_AB')
    ax.set_title(f'n={n} ring — Structural validity (B_i → A_{{i+1}} < 2·r_patch)')

    plt.tight_layout()
    plt.savefig(f"ring_n{n}_diagnostics.png", dpi=150, bbox_inches='tight')
    plt.show()


for n in N_RINGS:
    plot_ring_heatmaps(n)